# Lab type: review
# Course: DS203 — Feature Engineering & Pipelines
# Lesson: Choosing Which Features to Include
# Task: You have pre-computed feature importance scores and a trained model. Review the feature selection decision and evaluate whether the chosen features represent a good trade-off between importance and operational cost.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# Synthetic customer churn dataset
np.random.seed(42)
n_samples = 1000

data = {
    'tenure_days': np.random.uniform(1, 2000, n_samples),
    'monthly_spend': np.random.uniform(20, 200, n_samples),
    'support_tickets': np.random.poisson(2, n_samples),
    'last_login_days': np.random.exponential(10, n_samples),
    'contract_months': np.random.choice([1, 12, 24, 36], n_samples),
    'plan_paid': np.random.choice([0, 1], n_samples),
    'region_encoded': np.random.randint(0, 5, n_samples),
    'signup_month_sin': np.random.uniform(-1, 1, n_samples),
}

# Target: correlated with tenure, spend, and support (moderate signal)
churned = (
    (data['tenure_days'] < 300).astype(int) * 0.4 +
    (data['monthly_spend'] < 50).astype(int) * 0.3 +
    (data['support_tickets'] > 5).astype(int) * 0.3 +
    np.random.uniform(0, 0.5, n_samples)
)
churned = (churned > 0.5).astype(int)

X = pd.DataFrame(data)
y = pd.Series(churned, name='churned')

print(f"Dataset shape: {X.shape}")
print(f"Churn rate: {y.mean():.1%}")

In [ ]:
# Step 1: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Step 2: Compute feature importance via three methods

# Method 1: Univariate (SelectKBest with f_classif)
selector = SelectKBest(score_func=f_classif, k=len(X.columns))
selector.fit(X_train, y_train)
univariate_scores = selector.scores_

# Method 2: Tree-based importance
rf = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=5)
rf.fit(X_train, y_train)
tree_importances = rf.feature_importances_

# Method 3: Permutation importance (on test set)
perm_importance = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=42
)

# Combine into a DataFrame
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Univariate Score': univariate_scores,
    'Tree Importance': tree_importances,
    'Permutation Importance': perm_importance.importances_mean,
})

# Normalize scores to [0, 1] for comparison
for col in ['Univariate Score', 'Tree Importance', 'Permutation Importance']:
    min_val = importance_df[col].min()
    max_val = importance_df[col].max()
    if max_val > min_val:
        importance_df[col + ' (norm)'] = (importance_df[col] - min_val) / (max_val - min_val)
    else:
        importance_df[col + ' (norm)'] = 0

# Average across methods
norm_cols = [c for c in importance_df.columns if '(norm)' in c]
importance_df['Mean Importance'] = importance_df[norm_cols].mean(axis=1)
importance_df = importance_df.sort_values('Mean Importance', ascending=False)

print(importance_df[['Feature', 'Univariate Score', 'Tree Importance', 'Permutation Importance', 'Mean Importance']])

In [ ]:
# Step 3: Feature selection decision

# The data engineer decided: keep features in the top 70% of cumulative importance
cumsum = importance_df['Mean Importance'].cumsum()
cumsum_pct = cumsum / cumsum.iloc[-1]
selected_features = importance_df[cumsum_pct <= 0.70]['Feature'].tolist()

print(f"Features explaining 70% of cumulative importance: {selected_features}")
print(f"Number of features: {len(selected_features)}")
print(f"Cumulative importance threshold: {importance_df[cumsum_pct <= 0.70]['Mean Importance'].sum():.2f}")

## Review Question 1: Feature Selection Strategy

The selected features are: `tenure_days`, `monthly_spend`, `support_tickets`, and `contract_months`.

**Question:** Do you think this selection is justified? Why or why not?
- Consider: Are these features redundant with each other?
- Consider: Are any of these expensive or unstable to compute in production?
- Consider: Does dropping the remaining features lose meaningful signal?

*Write your answer below.*

In [ ]:
# YOUR ANSWER HERE (as a comment)
# This selection appears justified because...

## Review Question 2: Dropped Features

Look at the features that were dropped: `last_login_days`, `plan_paid`, `region_encoded`, `signup_month_sin`.

**Question:** For each dropped feature, evaluate whether dropping it was the right call:
1. Would you keep or drop `last_login_days`? (Low univariate importance but stable and cheap to compute)
2. Would you keep or drop `plan_paid`? (Low importance, but a business-critical variable)
3. Would you keep or drop `region_encoded`? (Low importance, but likely important for some customer segments)

*Write your reasoning below.*

In [ ]:
# YOUR ANSWER HERE (as comments)
# last_login_days: KEEP / DROP because...
# plan_paid: KEEP / DROP because...
# region_encoded: KEEP / DROP because...

## Review Question 3: Cost-Benefit Trade-off

Assume `plan_paid` is expensive to fetch from the database (requires a join to the billing system). But it has some predictive power (importance ≈ 0.05).

**Question:** Would you include `plan_paid` in the production pipeline? Justify your decision by considering:
- How much does adding one more feature slow down inference?
- How much does fetching it from the database cost operationally?
- Is the 0.05 importance threshold worth the extra latency and operational burden?

*Write your reasoning below.*

In [ ]:
# YOUR ANSWER HERE (as a comment)
# I would INCLUDE / EXCLUDE plan_paid because...

## Review Question 4: Testing the Selection

Build two models: one with the selected features, one with all features. Compare their performance.

**Question:** Does the selected feature set achieve comparable performance to using all features? If you lost significant performance by dropping features, what does that tell you about the selection decision?


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Model 1: All features
model_all = LogisticRegression(max_iter=1000, random_state=42)
model_all.fit(X_train, y_train)
auc_all = roc_auc_score(y_test, model_all.predict_proba(X_test)[:, 1])

# Model 2: Selected features
model_selected = LogisticRegression(max_iter=1000, random_state=42)
model_selected.fit(X_train[selected_features], y_train)
auc_selected = roc_auc_score(y_test, model_selected.predict_proba(X_test[selected_features])[:, 1])

print(f"AUC (all features): {auc_all:.4f}")
print(f"AUC (selected features): {auc_selected:.4f}")
print(f"Performance difference: {(auc_all - auc_selected):.4f}")
print(f"Features dropped: {len(X.columns) - len(selected_features)} out of {len(X.columns)}")

In [ ]:
# YOUR ANSWER HERE (as a comment)
# The performance difference is ___. This suggests that...
# If I were deploying this, I would...

## Review Question 5: AI-Generated Feature Selection

Imagine an AI tool generated the following code to select features:

```python
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(score_func=f_classif, k=3)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()
```

**Question:** What is wrong with this AI-generated code from a production perspective? (Hint: It's not the SelectKBest call itself.)

*Write your answer below.*

In [ ]:
# YOUR ANSWER HERE (as a comment)
# The problem is: ...